# Variables
- l = upper bound to the key length
- epsilon_sec = secrecy threshold -> (10^-9) in the paper
- epsilon_cor = correctness thresold -> fixed at 10^-15 as per paper
- a, b depends on the security analysis considered, for the paper a = 6, b = 19
- s_bn_l = lower bound on the events over the base b, with n photons
    - s_z1_l = lower bound of single photon events in base Z
- phi_b_u = upper bound on the phase error rate in base b (measurement error (for us e_z, e_x))
- lambda_ec = number of disclosed bit in error correction
- mu = [mu1, mu2, mu3] with mu3 = void and mu1 > mu2
- n = [n_z_mu1, n_z_mu2, n_z_mu3]
    - n_b_mu = number of detection in base b with intensity level mu
- tao_n = total probability of sending n photon state
- v_zn = number of error in z base when n phton are sent
- m_zk = number of error in z base when k intensity is sent
- p_k = [p_mu1, p_mu2, p_mu3] probability of sending a certain intensity
- p_kn = [p_mu1|n, p_mu2|n, p_mu3|n] probability of having intensity mu given n photon observed
- p_nk = [p_n|mu1, p_n|mu2, p_n|mu3] probability of reciving n phton given intensity mu

In [48]:
import math

In [49]:
# variables form the paper
param = {
"epsilon1": 1e-10, #hoefding epsilon for n_z_k
"epsilon2": 1e-10, #hoefding epsilon for m_x_k
"epsilon_cor": 1e-15,
"epsilon_sec": 1e-9,
"a": 6,
"b": 19
}

In [50]:
def entropy(probabilities):
    return -sum(p * math.log2(p) for p in probabilities if p > 0)

def bin_entropy(prob):
    p = [prob, 1-prob]
    return entropy(p)

# Functions

## extra function

In [51]:
# (A2)
def delta(value, epsilon = 1e-10):
    return math.sqrt((value * math.log10(1/epsilon))/2)

In [52]:
# (A21)
def lamda(a, b, c, d):
    factor1 = ((c+d)*(1-b)*b)/(c*d*math.log10(2))
    factor2 = ((c+d)/(c*d*(1-b)*b)) * (21**2/a**2)
    return factor1 * math.log2(factor2)

## basic functions

In [53]:
def tao(p_k, k):
    """tao = [tao_0, tao_1] where tao_n = total prob to send n photon state"""
    tao = []
    for n in range(0, 2):
        t = 0
        for i in range(0, len(k)):
            t += p_k[i] * ((math.e**(-k[i]) * k[i]**n) / math.factorial(n))
        tao.append(t)
    return tao

In [54]:
# (A4)
def hoefding_bound(value, epsilon = 1e-10):
    value_plus = value + delta(value, epsilon)
    value_minus = value - delta(value, epsilon)
    return value_plus, value_minus

In [55]:
# correction of (A18) with (3)
def hoefding_bound_correction(value, mu, p_mu,epsilon):
    value_plus, value_minus = hoefding_bound(value, epsilon)
    correction = math.e**mu / p_mu
    return value_plus * correction, value_minus * correction

## important function

In [56]:
# zero photon detection events lower bound
# with base z, equation (A19)
def s_z0_l(tao0, n_z, k, p, epsilon):
    mu1 = k[0]
    mu2 = k[1]
    # lower bound of detection events with intesity mu2 based on #detection
    # in z base
    _, n_zmu2_l = hoefding_bound_correction(n_z, mu2, p[1], epsilon)
    # upper bound of detection events with intesity mu2 based on #detection
    # in z base
    n_zmu1_u, _ = hoefding_bound_correction(n_z, mu1, p[0], epsilon)
    factor1 = tao0/(mu1 - mu2)
    term1 = mu1 * n_zmu2_l
    term2 = mu2 * n_zmu1_u
    return factor1 * (term1 - term2)

In [57]:
#equation (A14)
def s_z0_u(m_z, n_z, epsilon = 1e-10):
    return 2 * (m_z + delta(n_z, epsilon))

In [70]:
#(A17)
def s_z1_l(tao, k, p, m_z, n_z, epsilon):
    mu1 = k[0]
    mu2 = k[1]
    tao0 = tao[0]
    tao1 = tao[1]
    # lower bound of detection events with intesity mu2 based on #detection
    # in z base
    _, n_zmu2_l = hoefding_bound_correction(n_z, mu2, p[1], epsilon)
    # upper bound of detection events with intesity mu2 based on #detection
    # in z base
    n_zmu1_u, _ = hoefding_bound_correction(n_z, mu1, p[0], epsilon)
    sz0u = s_z0_u(m_z, n_z, epsilon)
    ## calculation
    factor1 = (tao1 * mu1)/(mu1 * (mu1- mu2))
    term1 = n_zmu2_l
    term2 = (mu2**2 / mu1**2) * n_zmu1_u
    term3 = (((mu1**2 - mu2**2)/mu1**2)*(sz0u/tao0))
    return factor1 * (term1 - term2 - term3)

In [59]:
# (A22)
def v_x1_u(tao1, m_x, k, p, epsilon):
    mu1 = k[0]
    mu2 = k[1]
    _, m_xmu2_l = hoefding_bound_correction(m_x, mu2, p[1], epsilon)
    m_xmu1_u, _ = hoefding_bound_correction(m_x, mu1, p[0], epsilon)
    factor1 = tao1 / (mu1 - mu2)
    return factor1 * (m_xmu1_u - m_xmu2_l)

In [60]:
# (A23)
# where s_x1_l = s_z1_l
def phi_z_u(v_x1_u, s_z1_l,epsilon_sec):
    l = lamda(epsilon_sec, v_x1_u / s_z1_l, s_z1_l, s_z1_l)
    return v_x1_u/s_z1_l * l

## main function

In [166]:
def length_upper(p, k, n, m, lamda_ec, param):
    epsilon1 = param["epsilon1"]
    epsilon2 = param["epsilon2"]
    epsilon_cor = param["epsilon_cor"]
    epsilon_sec = param["epsilon_sec"]
    a = param["a"]
    b = param["b"]
    t = tao(p, k)
    print(t[0])
    tao0 = t[0]
    tao1 = t[1]
    n_z = n[0]
    n_x = n[1]
    m_z = m[0]
    m_x = m[1]
    sz0l = s_z0_l(tao0, n_z, k, p, epsilon1)
    sz1l = s_z1_l(t, k, p, m_z, n_z, epsilon1)
    vx1u = v_x1_u(tao1, m_x, k, p, epsilon1)
    phizu = phi_z_u(vx1u, sz1l,epsilon_sec)
    factor1 = 1 - bin_entropy(phizu)
    term1 = a * math.log2(b/epsilon_sec)
    term2 = math.log2(2/epsilon_cor)
    l = sz0l + sz1l * factor1 - lamda_ec - term1 - term2
    return l


# File reading functions

In [62]:
import pandas as pd

# Mapping of encoded states to bases and bits
state_map = {
    '00': ('Z', 0),  # H
    '01': ('Z', 1),  # V
    '10': ('X', 0),  # D
    '11': ('X', 1)   # A
}

def read_states_limited(file_path, n):
    """Read up to n states from a file byte by byte and decode last 2 bits."""
    states = []
    with open(file_path, 'rb') as f:
        for _ in range(n):
            byte = f.read(1)
            if not byte:
                break  # EOF
            bit_str = format(byte[0], '08b')[-2:]
            states.append(bit_str)
    return states

def read_bitpos_limited(file_path, n):
    """Read bit positions from CSV and return up to n absolute positions."""
    pos_data = pd.read_csv(file_path, header=None)[0].tolist()
    abs_positions = []
    i = 0
    while i < len(pos_data) and len(abs_positions) < n:
        if pos_data[i] == -1:
            abs_pos = pos_data[i+1]
            abs_positions.append(abs_pos)
            for j in range(2, 24002):
                if i+j >= len(pos_data) or len(abs_positions) >= n:
                    break
                abs_pos += pos_data[i+j]
                abs_positions.append(abs_pos)
            i += 24002
        else:
            i += 1
    return abs_positions[:n]

def analyze_quantum_files_limited(states_file, decoy_file, states_rcv_file, bitpos_file, n):
    # Read up to n states from each file
    alice_states = read_states_limited(states_file, n)
    rcv_states = read_states_limited(states_rcv_file, n)
    decoys = read_states_limited(decoy_file, n)  # still read, even if unused
    positions = read_bitpos_limited(bitpos_file, n)

    min_len = min(len(alice_states), len(rcv_states), len(positions))
    alice_states = alice_states[:min_len]
    rcv_states = rcv_states[:min_len]

    # Counters
    counts = {'Z': 0, 'X': 0}
    errors = {'Z': 0, 'X': 0}
    alice_counts = {'Z': 0, 'X': 0}

    for a, r in zip(alice_states, rcv_states):
        alice_basis, alice_bit = state_map[a]
        rcv_basis, rcv_bit = state_map[r]
        alice_counts[alice_basis] += 1
        if alice_basis == rcv_basis: # if measured with the same 
            counts[alice_basis] += 1
            if alice_bit != rcv_bit: # if there is no error
                errors[alice_basis] += 1

    total = len(alice_states)
    probabilities = {basis: alice_counts[basis]/total for basis in ['Z', 'X']}
    error_rates = {basis: (errors[basis]/counts[basis] if counts[basis] > 0 else 0) 
                   for basis in ['Z', 'X']}

    return {
        'error_detections': errors,
        'total_detections': counts,
        'alice_probabilities': probabilities
    }

# Example usage:
# result = analyze_quantum_files_limited('states.txt', 'decoy.txt', 'statesRCV.txt', 'bitpos.txt', n=50000)
# print(result)


# data collection

In [155]:
k = [0.6 , 0.17]
limit = 1000
path = "QCS_2025/"
states_path = path + "states.txt"
decoy_path = path + "decoy.txt"
statesRCV_path = path + "statesRCV.txt"
bitpos_path = path + "bitpos.txt"

limited data collection for simple testing

In [157]:
data = analyze_quantum_files_limited(states_path, decoy_path, statesRCV_path, bitpos_path, limit)

In [146]:
print(data)

{'error_detections': {'Z': 420, 'X': 38}, 'total_detections': {'Z': 20780, 'X': 2466}, 'alice_probabilities': {'Z': 0.90846, 'X': 0.09154}}


In [187]:
p = [data["alice_probabilities"]["Z"], data["alice_probabilities"]["X"]]
n = [data["total_detections"]["Z"], data["total_detections"]["X"]]
m = [data["error_detections"]["Z"], data["error_detections"]["X"]+1]
print(p)
print(n)
print(m)

[0.908, 0.092]
[415, 50]
[12, 1]


In [186]:
l = length_upper(p, k, n, m, 10, param)
print(l)

0.5759381287002434
1979.8616049906018
